In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [2]:
customers_df = pd.read_csv("Customers1.csv")

In [3]:
customers_df["DOB"] = pd.to_datetime(customers_df["DOB"])

today = pd.Timestamp.today()

customers_df["Age"] = (
    (today - customers_df["DOB"]).dt.days // 365
)

In [4]:
eligible_customers = customers_df[
    customers_df["Age"] >= 21
].copy()

In [5]:
insurance_customers = eligible_customers.sample(
    n=3000,
    random_state=42
).reset_index(drop=True)

In [6]:
insurance_customers.shape

(3000, 15)

In [7]:
POLICY_STATUS = [

    "Active",
    "Expired"

]

POLICY_STATUS_WEIGHTS = [

    80,
    20

]

In [8]:
insurance = []

policy_number = 1

In [9]:
for _, customer in insurance_customers.iterrows():

    customer_id = customer["Customer_ID"]

    age = customer["Age"]

    income = customer["Annual_Income"]

    marital_status = customer["Marital_Status"]

    join_date = pd.to_datetime(customer["Join_Date"])

    # --------------------------
    # Choose Policy Type
    # --------------------------

    if age <= 30:

        policy_type = random.choices(

            ["Vehicle", "Health"],

            weights=[60, 40],

            k=1

        )[0]

    elif age <= 50:

        if marital_status == "Married":

            policy_type = random.choices(

                ["Health", "Life", "Vehicle"],

                weights=[40, 40, 20],

                k=1

            )[0]

        else:

            policy_type = random.choices(

                ["Health", "Vehicle"],

                weights=[60, 40],

                k=1

            )[0]

    else:

        policy_type = random.choices(

            ["Life", "Health"],

            weights=[70, 30],

            k=1

        )[0]

    # --------------------------
    # Coverage Amount
    # --------------------------

    if policy_type == "Vehicle":

        coverage = random.randint(300000, 1000000)

    elif policy_type == "Health":

        coverage = random.randint(500000, 2000000)

    else:

        coverage = random.randint(1000000, 10000000)

    # --------------------------
    # Premium
    # --------------------------

    premium = round(

        coverage * random.uniform(0.01, 0.03),

        2

    )

    # --------------------------
    # Dates
    # --------------------------

    start_date = fake.date_between(

        start_date=join_date.date(),

        end_date="today"

    )

    end_date = (

        pd.to_datetime(start_date)

        + pd.DateOffset(years=1)

    )

    # --------------------------
    # Status
    # --------------------------

    policy_status = random.choices(

        POLICY_STATUS,

        weights=POLICY_STATUS_WEIGHTS,

        k=1

    )[0]

    policy_id = f"P{policy_number:06d}"

    policy_number += 1

    insurance.append({

        "Policy_ID": policy_id,

        "Customer_ID": customer_id,

        "Policy_Type": policy_type,

        "Premium": premium,

        "Coverage": coverage,

        "Start_Date": start_date,

        "End_Date": end_date,

        "Policy_Status": policy_status

    })

In [10]:
insurance_df = pd.DataFrame(insurance)

insurance_df.shape

(3000, 8)

In [11]:
insurance_df.head()

,Policy_ID,Customer_ID,Policy_Type,Premium,Coverage,Start_Date,End_Date,Policy_Status
0,P000001,C06253,Life,35250.35,1419610,2024-03-03,2025-03-03,Active
1,P000002,C04685,Life,64002.64,2719583,2026-05-20,2027-05-20,Expired
2,P000003,C01732,Health,14728.61,1384834,2022-03-10,2023-03-10,Active
3,P000004,C04743,Life,30674.21,1445199,2023-05-05,2024-05-05,Active
4,P000005,C04522,Health,19881.22,1379796,2025-09-10,2026-09-10,Active


In [12]:
insurance_df.isnull().sum()

Policy_ID        0
Customer_ID      0
Policy_Type      0
Premium          0
Coverage         0
Start_Date       0
End_Date         0
Policy_Status    0
dtype: int64

In [13]:
insurance_df["Policy_ID"].duplicated().sum()

np.int64(0)

In [14]:
insurance_df["Policy_Type"].value_counts()

Policy_Type
Health     1161
Life       1090
Vehicle     749
Name: count, dtype: int64

In [15]:
insurance_df["Policy_Status"].value_counts(normalize=True) * 100

Policy_Status
Active     78.366667
Expired    21.633333
Name: proportion, dtype: float64

In [16]:
(insurance_df["Premium"] > 0).all()

np.True_

In [17]:
insurance_df["Start_Date"] = pd.to_datetime(insurance_df["Start_Date"])
insurance_df["End_Date"] = pd.to_datetime(insurance_df["End_Date"])

insurance_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Policy_ID      3000 non-null   object        
 1   Customer_ID    3000 non-null   object        
 2   Policy_Type    3000 non-null   object        
 3   Premium        3000 non-null   float64       
 4   Coverage       3000 non-null   int64         
 5   Start_Date     3000 non-null   datetime64[ns]
 6   End_Date       3000 non-null   datetime64[ns]
 7   Policy_Status  3000 non-null   object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(4)
memory usage: 187.6+ KB


In [18]:
insurance_df.to_csv(
    "Insurance.csv",
    index=False
)